<a href="https://colab.research.google.com/github/robertnathe/AutonomousCodingAgent/blob/main/Copy_of_JEPA_for_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, pickle, tarfile, requests
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---------- Download (unchanged) ----------
def download_cifar10(data_dir="cifar10_data"):
    base_url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    filename = base_url.split("/")[-1]
    filepath = os.path.join(data_dir, filename)
    extract_dir = os.path.join(data_dir, 'cifar-10-batches-py')
    if not os.path.exists(extract_dir):
        os.makedirs(data_dir, exist_ok=True)
        print(f"Downloading CIFAR-10 to {filepath}...")
        response = requests.get(base_url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        with open(filepath, 'wb') as f, tqdm(total=total_size, unit='iB', unit_scale=True) as bar:
            for data in response.iter_content(1024):
                bar.update(len(data))
                f.write(data)
        with tarfile.open(filepath, 'r:gz') as tar:
            tar.extractall(path=data_dir)
    return extract_dir

def unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

data_dir = download_cifar10()
train_data, train_labels = [], []
for i in range(1,6):
    batch = unpickle(os.path.join(data_dir, f'data_batch_{i}'))
    train_data.append(batch[b'data'])
    train_labels.extend(batch[b'labels'])
train_data = np.concatenate(train_data).reshape(-1,3,32,32).astype(np.float32) / 255.0
train_data = torch.tensor(train_data)   # keep in float [0,1]; normalization done in transform

# ---------- JEPA Dataset ----------
class JEPADataset(torch.utils.data.Dataset):
    def __init__(self, data, patch_size=4, block_size=2, num_blocks=4):
        self.data = data
        self.P = patch_size
        self.grid = 32 // self.P
        self.num_patches = self.grid**2
        self.block_size = block_size
        self.num_blocks = num_blocks
        # CIFAR-10 normalization (applied after patch extraction for simplicity)
        self.mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1)
        self.std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(1,3,1,1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]  # (3,32,32) in [0,1]
        # Normalize whole image
        img = (img - self.mean) / self.std
        # Patchify
        patches = img.unfold(1, self.P, self.P).unfold(2, self.P, self.P)
        patches = patches.contiguous().view(3, self.grid, self.grid, self.P, self.P)
        patches = patches.permute(1,2,0,3,4).reshape(self.num_patches, 3, self.P, self.P)

        # Target block selection (non-overlapping)
        target_pos = []
        occupied = np.zeros((self.grid, self.grid), dtype=bool)
        for _ in range(self.num_blocks):
            for attempt in range(100):
                r = np.random.randint(0, self.grid - self.block_size + 1)
                c = np.random.randint(0, self.grid - self.block_size + 1)
                if not occupied[r:r+self.block_size, c:c+self.block_size].any():
                    occupied[r:r+self.block_size, c:c+self.block_size] = True
                    for dr in range(self.block_size):
                        for dc in range(self.block_size):
                            target_pos.append((r+dr)*self.grid + (c+dc))
                    break
        target_pos = torch.tensor(target_pos, dtype=torch.long)
        all_pos = torch.arange(self.num_patches)
        context_pos = all_pos[~torch.isin(all_pos, target_pos)]

        return patches[context_pos], patches[target_pos], context_pos, target_pos

def jepa_collate(batch):
    """Pads variable‑length context/target patches across a batch."""
    context_list, target_list, ctx_pos_list, tgt_pos_list = zip(*batch)
    max_ctx = max(c.shape[0] for c in context_list)
    max_tgt = max(t.shape[0] for t in target_list)
    B = len(batch)
    P = context_list[0].shape[-1]   # patch size

    ctx_padded = torch.zeros(B, max_ctx, 3, P, P)
    tgt_padded = torch.zeros(B, max_tgt, 3, P, P)
    ctx_pos_pad = torch.zeros(B, max_ctx, dtype=torch.long)
    tgt_pos_pad = torch.zeros(B, max_tgt, dtype=torch.long)

    for i in range(B):
        n_ctx = context_list[i].size(0)
        n_tgt = target_list[i].size(0)
        ctx_padded[i, :n_ctx] = context_list[i]
        tgt_padded[i, :n_tgt] = target_list[i]
        ctx_pos_pad[i, :n_ctx] = ctx_pos_list[i]
        tgt_pos_pad[i, :n_tgt] = tgt_pos_list[i]
    return ctx_padded, tgt_padded, ctx_pos_pad, tgt_pos_pad

# ---------- Models ----------
class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, in_chans=3, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B*N, C, H, W)
        return self.proj(x).flatten(2).transpose(1,2)  # (B*N, 1, embed_dim)

def get_sinusoidal_pe(num_patches, embed_dim):
    pe = torch.zeros(num_patches, embed_dim)
    position = torch.arange(num_patches).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0)/embed_dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class ContextEncoder(nn.Module):
    def __init__(self, patch_size=4, embed_dim=192, depth=4, num_heads=3):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, 3, embed_dim)
        self.pos_embed = nn.Parameter(get_sinusoidal_pe(64, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

    def forward(self, patches, positions):
        B, N, C, H, W = patches.shape
        patches_flat = patches.reshape(B*N, C, H, W)
        emb = self.patch_embed(patches_flat).squeeze(1).reshape(B, N, -1)
        emb += self.pos_embed[positions]
        return self.transformer(emb)

class TargetEncoder(nn.Module):
    def __init__(self, patch_size=4, embed_dim=192, depth=2, num_heads=3):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, 3, embed_dim)
        self.pos_embed = nn.Parameter(get_sinusoidal_pe(64, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

    def forward(self, patches, positions):
        B, N, C, H, W = patches.shape
        patches_flat = patches.reshape(B*N, C, H, W)
        emb = self.patch_embed(patches_flat).squeeze(1).reshape(B, N, -1)
        emb += self.pos_embed[positions]
        return self.transformer(emb)

class Predictor(nn.Module):
    def __init__(self, embed_dim=192, depth=2, num_heads=3):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        # pos_embed will be set externally

    def forward(self, context_latent, target_positions):
        B, N_t = target_positions.shape
        mask_tokens = self.mask_token.expand(B, N_t, -1)
        pos = self.pos_embed[target_positions]
        target_queries = mask_tokens + pos
        combined = torch.cat([context_latent, target_queries], dim=1)
        out = self.transformer(combined)
        return out[:, -N_t:, :]

def update_target_encoder(ctx_enc, tgt_enc, momentum=0.996):
    with torch.no_grad():
        for p_ctx, p_tgt in zip(ctx_enc.parameters(), tgt_enc.parameters()):
            p_tgt.data = p_tgt.data * momentum + p_ctx.data * (1 - momentum)

# ---------- Pre‑training ----------
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = JEPADataset(train_data)
    loader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=jepa_collate, num_workers=2)

    ctx_enc = ContextEncoder().to(device)
    tgt_enc = TargetEncoder().to(device)
    pred = Predictor().to(device)
    pred.pos_embed = tgt_enc.pos_embed

    tgt_enc.load_state_dict(ctx_enc.state_dict())
    optimizer = torch.optim.AdamW(list(ctx_enc.parameters())+list(pred.parameters()), lr=1e-3, weight_decay=0.04)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    for epoch in range(100):
        ctx_enc.train(); pred.train(); tgt_enc.eval()
        total_loss = 0
        for ctx_p, tgt_p, ctx_pos, tgt_pos in loader:
            ctx_p = ctx_p.to(device); tgt_p = tgt_p.to(device)
            tgt_pos = tgt_pos.to(device)
            # context encoding
            c_latent = ctx_enc(ctx_p, ctx_pos)
            # target encoding (no grad)
            with torch.no_grad():
                t_latent = tgt_enc(tgt_p, tgt_pos)
            # prediction
            p_latent = pred(c_latent, tgt_pos)
            loss = F.mse_loss(p_latent, t_latent)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            update_target_encoder(ctx_enc, tgt_enc)
            total_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch+1:3d} | Loss {total_loss/len(loader):.6f}")

    torch.save({'context_encoder': ctx_enc.state_dict(), 'target_encoder': tgt_enc.state_dict()}, 'jepa_pretrained.pth')


  0%|          | 472k/170M [00:20<2:31:08, 18.7kiB/s]

JEPA cifar-10 dataset

In [ ]:
import os, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

def unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

script_dir = os.path.dirname(os.path.abspath(__file__))
data_dir = os.path.join(script_dir, "cifar-10-batches-py")

train_data, train_labels = [], []
for i in range(1, 6):
    batch = unpickle(os.path.join(data_dir, f"data_batch_{i}"))
    train_data.append(batch[b'data'])
    train_labels.extend(batch[b'labels'])

train_data = np.concatenate(train_data).reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
train_data = torch.tensor(train_data)

class JEPADataset(torch.utils.data.Dataset):
    def __init__(self, data, patch_size=4, block_size=2, num_blocks=4):
        self.data = data
        self.P = patch_size
        self.grid = 32 // self.P
        self.num_patches = self.grid ** 2
        self.block_size = block_size
        self.num_blocks = num_blocks
        self.mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
        self.std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]
        img = (img - self.mean) / self.std
        patches = img.unfold(1, self.P, self.P).unfold(2, self.P, self.P)
        patches = patches.contiguous().view(3, self.grid, self.grid, self.P, self.P)
        patches = patches.permute(1, 2, 0, 3, 4).reshape(self.num_patches, 3, self.P, self.P)
        target_pos = []
        occupied = np.zeros((self.grid, self.grid), dtype=bool)
        for _ in range(self.num_blocks):
            for attempt in range(100):
                r = np.random.randint(0, self.grid - self.block_size + 1)
                c = np.random.randint(0, self.grid - self.block_size + 1)
                if not occupied[r:r + self.block_size, c:c + self.block_size].any():
                    occupied[r:r + self.block_size, c:c + self.block_size] = True
                    for dr in range(self.block_size):
                        for dc in range(self.block_size):
                            target_pos.append((r + dr) * self.grid + (c + dc))
                    break
        target_pos = torch.tensor(target_pos, dtype=torch.long)
        all_pos = torch.arange(self.num_patches)
        context_pos = all_pos[~torch.isin(all_pos, target_pos)]
        return patches[context_pos], patches[target_pos], context_pos, target_pos

def jepa_collate(batch):
    context_list, target_list, ctx_pos_list, tgt_pos_list = zip(*batch)
    max_ctx = max(c.shape[0] for c in context_list)   # number of context patches
    max_tgt = max(t.shape[0] for t in target_list)    # number of target patches
    B = len(batch)
    P = context_list[0].shape[-1]                     # patch size (last dimension)
    ctx_padded = torch.zeros(B, max_ctx, 3, P, P)
    tgt_padded = torch.zeros(B, max_tgt, 3, P, P)
    ctx_pos_pad = torch.zeros(B, max_ctx, dtype=torch.long)
    tgt_pos_pad = torch.zeros(B, max_tgt, dtype=torch.long)
    for i in range(B):
        n_ctx = context_list[i].size(0)
        n_tgt = target_list[i].size(0)
        ctx_padded[i, :n_ctx] = context_list[i]
        tgt_padded[i, :n_tgt] = target_list[i]
        ctx_pos_pad[i, :n_ctx] = ctx_pos_list[i]
        tgt_pos_pad[i, :n_tgt] = tgt_pos_list[i]
    return ctx_padded, tgt_padded, ctx_pos_pad, tgt_pos_pad

class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, in_chans=3, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

def get_sinusoidal_pe(num_patches, embed_dim):
    pe = torch.zeros(num_patches, embed_dim)
    position = torch.arange(num_patches).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class ContextEncoder(nn.Module):
    def __init__(self, patch_size=4, embed_dim=192, depth=4, num_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, 3, embed_dim)
        self.pos_embed = nn.Parameter(get_sinusoidal_pe(64, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

    def forward(self, patches, positions):
        B, N, C, H, W = patches.shape
        patches_flat = patches.reshape(B * N, C, H, W)
        emb = self.patch_embed(patches_flat).squeeze(1).reshape(B, N, -1)
        emb = emb + self.pos_embed[positions]
        return self.transformer(emb)

class TargetEncoder(nn.Module):
    def __init__(self, patch_size=4, embed_dim=192, depth=4, num_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, 3, embed_dim)
        self.pos_embed = nn.Parameter(get_sinusoidal_pe(64, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

    def forward(self, patches, positions):
        B, N, C, H, W = patches.shape
        patches_flat = patches.reshape(B * N, C, H, W)
        emb = self.patch_embed(patches_flat).squeeze(1).reshape(B, N, -1)
        emb = emb + self.pos_embed[positions]
        return self.transformer(emb)

class Predictor(nn.Module):
    def __init__(self, embed_dim=192, depth=2, num_heads=4):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(get_sinusoidal_pe(64, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

    def forward(self, context_latent, target_positions):
        B, N_t = target_positions.shape
        mask_tokens = self.mask_token.expand(B, N_t, -1)
        pos = self.pos_embed[target_positions]
        target_queries = mask_tokens + pos
        combined = torch.cat([context_latent, target_queries], dim=1)
        out = self.transformer(combined)
        return out[:, -N_t:, :]

def update_target_encoder(ctx_enc, tgt_enc, momentum=0.996):
    with torch.no_grad():
        for p_ctx, p_tgt in zip(ctx_enc.parameters(), tgt_enc.parameters()):
            p_tgt.data.mul_(momentum).add_(p_ctx.data, alpha=1 - momentum)

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = JEPADataset(train_data)
    loader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=jepa_collate, num_workers=2)
    ctx_enc = ContextEncoder().to(device)
    tgt_enc = TargetEncoder().to(device)
    pred = Predictor().to(device)
    pred.pos_embed = tgt_enc.pos_embed
    tgt_enc.load_state_dict(ctx_enc.state_dict(), strict=False)
    optimizer = torch.optim.AdamW(list(ctx_enc.parameters()) + list(pred.parameters()), lr=1e-3, weight_decay=0.04)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    for epoch in range(100):
        ctx_enc.train()
        pred.train()
        tgt_enc.eval()
        total_loss = 0.0
        for ctx_p, tgt_p, ctx_pos, tgt_pos in loader:
            ctx_p = ctx_p.to(device)
            tgt_p = tgt_p.to(device)
            ctx_pos = ctx_pos.to(device)
            tgt_pos = tgt_pos.to(device)
            c_latent = ctx_enc(ctx_p, ctx_pos)
            with torch.no_grad():
                t_latent = tgt_enc(tgt_p, tgt_pos)
            p_latent = pred(c_latent, tgt_pos)
            loss = F.mse_loss(p_latent, t_latent)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            update_target_encoder(ctx_enc, tgt_enc)
            total_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch + 1:3d} | Loss {total_loss / len(loader):.6f}")
    torch.save(
        {'context_encoder': ctx_enc.state_dict(), 'target_encoder': tgt_enc.state_dict()},
        'jepa_pretrained.pth'
    )
